# Notebook 09 — Custom Material Models and USERMAT

## What you will learn
1. Standard material models in ANSYS (elastic, BISO, MISO, Chaboche)
2. When and why you need a custom USERMAT (Fortran subroutine)
3. How to compile and load a USERMAT DLL
4. How to define and calibrate a Chaboche cyclic plasticity model
5. Temperature-dependent material properties

---

## Part 1: The Constitutive Framework

### Strain decomposition

The total strain tensor is decomposed into elastic and plastic parts:

$$\boldsymbol{\varepsilon} = \boldsymbol{\varepsilon}^{el} + \boldsymbol{\varepsilon}^{pl}$$

Elastic part (Hooke's law):
$$\boldsymbol{\sigma} = [C^{el}] : \boldsymbol{\varepsilon}^{el} = [C^{el}] : (\boldsymbol{\varepsilon} - \boldsymbol{\varepsilon}^{pl})$$

### Yield function (von Mises)

$$f(\boldsymbol{\sigma}, \mathbf{X}, R) = \bar{\sigma}(\boldsymbol{\sigma} - \mathbf{X}) - \sigma_y^0 - R(p) \leq 0$$

where:
- $\bar{\sigma}$ = von Mises effective stress (relative to backstress $\mathbf{X}$)
- $\mathbf{X}$ = kinematic backstress (from Chaboche model)
- $R(p)$ = isotropic hardening variable
- $p$ = accumulated plastic strain = $\int_0^t \dot{p} \, dt'$

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt

# ── Bilinear vs multilinear stress-strain curves ───────────────────────────
# Demonstrate the material models available in the toolbox

E    = 200e9    # Young's modulus (Pa)
nu   = 0.30
sig_y = 250e6   # Initial yield stress (Pa)
E_T   = 2e9     # Tangent modulus (Pa) = 1% of E (typical)

# Strain range
eps = np.linspace(0, 0.03, 300)

# Bilinear isotropic hardening (BISO)
eps_y = sig_y / E
sig_biso = np.where(
    eps <= eps_y,
    E * eps,
    sig_y + E_T * (eps - eps_y)
)

# Multilinear (MISO) — piecewise from experimental data
eps_miso  = np.array([0.00125, 0.005, 0.010, 0.020, 0.030])
sig_miso  = np.array([250e6,   280e6,  310e6,  350e6,  380e6])
sig_miso_interp = np.interp(eps, np.concatenate([[0, eps_y], eps_miso]),
                               np.concatenate([[0, sig_y], sig_miso]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(eps * 100, sig_biso / 1e6, 'b-', linewidth=2, label=f'BISO (E_T={E_T/1e9:.0f} GPa)')
ax1.plot(eps_miso * 100, sig_miso / 1e6, 'ro-', linewidth=2, markersize=6, label='MISO data points')
ax1.plot(eps * 100, sig_miso_interp / 1e6, 'r--', linewidth=1, alpha=0.6, label='MISO interpolated')
ax1.axvline(eps_y * 100, color='gray', linestyle=':', label=f'ε_y = {eps_y*100:.3f}%')
ax1.set_xlabel('Strain (%)'); ax1.set_ylabel('Stress (MPa)')
ax1.set_title('Stress-Strain: BISO vs MISO'); ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

# Chaboche cyclic hysteresis loop simulation
# Simplified 1-backstress Chaboche
C = 20000e6    # kinematic hardening modulus
gamma = 200    # dynamic recovery

def chaboche_step(sig, alpha, eps_prev, deps, E, sig_y):
    """One integration step of Chaboche kinematic hardening (1D, 1 backstress)."""
    sig_trial = sig + E * deps
    f_trial = abs(sig_trial - alpha) - sig_y
    if f_trial <= 0:
        return sig_trial, alpha
    # Plastic corrector
    n = np.sign(sig_trial - alpha)
    d_eps_pl = f_trial / (E + C - gamma * abs(alpha) * n)
    sig_new   = sig_trial - E * n * d_eps_pl
    alpha_new = alpha + (C - gamma * alpha * n) * d_eps_pl
    return sig_new, alpha_new

# Simulate a cyclic strain history
eps_max = 0.015
n_cycles = 3
n_pts    = 100

eps_history = []
sig_history = []
for cycle in range(n_cycles):
    eps_history.extend(np.linspace(0, eps_max, n_pts))
    eps_history.extend(np.linspace(eps_max, -eps_max, n_pts))
    eps_history.extend(np.linspace(-eps_max, 0, n_pts))

sig, alpha = 0.0, 0.0
sigs = [sig]
eps_arr = np.array(eps_history)
for i in range(1, len(eps_arr)):
    deps = eps_arr[i] - eps_arr[i-1]
    sig, alpha = chaboche_step(sig, alpha, eps_arr[i-1], deps, E, sig_y)
    sigs.append(sig)

ax2.plot(np.array(eps_arr) * 100, np.array(sigs) / 1e6, 'b-', linewidth=1)
ax2.axhline(sig_y/1e6, color='gray', linestyle=':', alpha=0.5)
ax2.axhline(-sig_y/1e6, color='gray', linestyle=':', alpha=0.5)
ax2.set_xlabel('Strain (%)'); ax2.set_ylabel('Stress (MPa)')
ax2.set_title(f'Chaboche Cyclic Hardening ({n_cycles} cycles)'); ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print('Note the Bauschinger effect: yield on reversal is reduced from σ_y = 250 MPa')
print('as the backstress α accumulates opposite the original loading direction.')

In [ ]:
# ── Apply materials using the toolbox API ─────────────────────────────────
from ams.materials.standard import assign_elastic, assign_bilinear_plastic, assign_chaboche

# Show function signatures (no live MAPDL needed)
print('assign_elastic:')
print(assign_elastic.__doc__[:500])
print('\n---')
print('assign_bilinear_plastic:')
print(assign_bilinear_plastic.__doc__[:500])

---

## Part 2: USERMAT — Custom Fortran Material Models

ANSYS USERMAT is a Fortran subroutine interface that lets you define any
constitutive model that ANSYS doesn't have built-in.

### When you need USERMAT

- Growth mechanics (biological tissue growth tensor $\mathbf{F}_g$)
- Phase-field damage models
- Multi-scale homogenization (FE²)
- Custom neural constitutive models (NCM, from the sister toolbox)
- Any material with internal state variables beyond what TB,USER provides

### USERMAT subroutine interface

The Fortran subroutine signature is fixed by ANSYS:

```fortran
SUBROUTINE usermat(
    matId, elemId, kDomIntPt, kLayer, kSectPt,
    ldstep, isubst, keycut,
    nDirect, nShear, ncomp, nStatev, nProp,
    Time, dTime, Temp, dTemp,
    stress, ustatev, dsdePl, sedEl, sedPl, epseq,
    Strain, dStrain, epsPl,
    prop, coords, rotateM, defGrad, predef,
    npredef)
```

Key arrays:
- `stress(ncomp)`: stress vector (updated each call)
- `ustatev(nStatev)`: internal state variables (e.g., plastic strain, backstress)
- `dStrain(ncomp)`: strain increment for this time step
- `prop(nProp)`: material parameters from TB,USER,matId → TBDATA

### Compilation on Windows

ANSYS ships a batch file for compiling custom usermats:
```cmd
D:\ANSYS Inc\v252\ansys\custom\user\winx64\ANSUSERSHARED.BAT
```

This compiles your `usermat.F` using the Intel Fortran compiler bundled
with ANSYS and produces `usermatLib.dll`.

Set `ANS_USER_PATH` to the directory containing the DLL before
launching MAPDL:
```python
import os
os.environ['ANS_USER_PATH'] = r'D:\Projects\usermat\build'
```

### Reference

Full USERMAT documentation with working Fortran examples is in
`D:/Projects/MAPDL/extensions/usermat_extended.F` and
`D:/Projects/MAPDL/reference/USERMAT_REFERENCE.md`.

---

## Summary

| Model | MAPDL command | n_params | When to use |
|-------|--------------|----------|-------------|
| Elastic | MP,EX+PRXY | 2 | All elastic problems |
| BISO | TB,BISO + TBDATA | 2 (σ_y, E_T) | Monotonic plasticity |
| MISO | TB,MISO + TBDATA | 2×N | Nonlinear hardening curve |
| Chaboche | TB,CHABOCHE + NLISO | 2k+2 | Cyclic loading / fatigue |
| Neo-Hookean | TB,HYPER,NEO | 2 (μ, d₁) | Rubber / soft tissue |
| USERMAT | TB,USER + custom DLL | N | Any custom model |

**Next:** The full toolbox guide in `ANSYS_SIM_TOOLBOX_GUIDE.md`